<a href="https://colab.research.google.com/github/Lucianadeoliveira/gaTE-lab/blob/main/FRALCAT%20/ANALYSIS_AND_PROTEIN_INFERENCE/TED_CATH_extract_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This script integrates protein information from UniProt, TED, and CATH databases.
It retrieves structural predictions and classification data for a list of UniProt IDs,
and appends these annotations to an existing protein information table.

Workflow:
1. Read a list of UniProt IDs from an input CSV file.
2. Query the TED API to retrieve structural predictions and metadata (TED ID, taxonomy, CATH label, pLDDT score).
3. Query the CATH database to retrieve classification details (name and description) for the associated CATH label.
4. Append the retrieved information to the existing UniProt table and save it into a new CSV file.

This integration allows systematic enrichment of protein annotations with structure-based information.


In [7]:
import requests
import csv
from bs4 import BeautifulSoup
from datetime import datetime
import os

# Input CSV file
csv_file = '/content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste1/information_proteins_uniprot.csv'

# ==============================
# Functions
# ==============================
def get_ted_data(uniprot_id):
    """Fetch TED data from the CATH TED API for a given UniProt ID."""
    url = f'https://ted.cathdb.info/api/v1/uniprot/summary/{uniprot_id}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['count'] > 0:
            return data['data'][0]
    return None

def get_cath_classification_info(cath_label):
    """Fetch CATH classification name and description from CATH website using a CATH label."""
    url = f'https://www.cathdb.info/version/latest/superfamily/{cath_label}/classification'
    response = requests.get(url)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        name_tag = soup.find('h1')
        desc_tag = soup.find('p')
        classification_name = name_tag.text.strip() if name_tag else "No classification name"
        classification_description = desc_tag.text.strip() if desc_tag else "No description"
        return classification_name, classification_description

    return "Not Found", "Not Found"

def process_uniprot_codes():
    """Update the UniProt CSV file in place with TED and CATH information."""
    update_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Read existing CSV
    with open(csv_file, 'r', newline='') as infile:
        reader = csv.DictReader(infile, delimiter='\t')
        fieldnames = [f for f in reader.fieldnames if f is not None]

        # Add new columns if not already present
        new_cols = [
            'TED ID',
            'Taxonomy',
            'CATH Label',
            'PLDDT',
            'CATH Classification Name',
            'CATH Classification Description',
            'Last Update'
        ]
        for col in new_cols:
            if col not in fieldnames:
                fieldnames.append(col)

        rows = []
        for row in reader:
            row = {k: v for k, v in row.items() if k is not None}
            uniprot_id = row[reader.fieldnames[0]]
            ted_data = get_ted_data(uniprot_id)

            if ted_data:
                row['TED ID'] = ted_data['ted_id']
                row['Taxonomy'] = ted_data['tax_scientific_name']
                row['CATH Label'] = ted_data['cath_label']
                row['PLDDT'] = ted_data['plddt']

                cath_name, cath_description = get_cath_classification_info(ted_data['cath_label'])
                row['CATH Classification Name'] = cath_name
                row['CATH Classification Description'] = cath_description
            else:
                for col in new_cols[:-1]:  # all except 'Last Update'
                    row[col] = "Not found"

            row['Last Update'] = update_time
            rows.append(row)

    # Save directly to the same CSV file
    with open(csv_file, 'w', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter='\t')
        writer.writeheader()
        writer.writerows(rows)

# Run
process_uniprot_codes()
